# 5.6 · 决策树分类器 / Decision Tree Classifier

> **课程定位 / Where this fits**
> 第 6 课，**Part 5 · 监督学习：分类**。
> Lesson 6, **Part 5 · Supervised Classification**.
>
> 4.11 讲过回归树，这一课是分类版。决策树用一连串"是/否"问题把特征空间切成**轴对齐的方块**，每块投一个类别。它**可解释性极强**（能画成流程图）、不需缩放、能处理混合类型特征——更重要的是，它是**随机森林(5.7)、GBDT(5.8)、XGBoost(5.9)** 的基石。
> 4.11 covered regression trees; this is the classification version. A decision tree splits feature space into **axis-aligned boxes** via a series of yes/no questions, voting one class per box. It is **highly interpretable** (drawable as a flowchart), needs no scaling, handles mixed feature types — and crucially is the building block of **Random Forest (5.7), GBDT (5.8), XGBoost (5.9)**.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - 节点 node —— 一组样本 / a set of samples
> - $p_k$ —— 节点中类别 $k$ 的比例 / fraction of class $k$ in a node
> - Gini $=1-\sum_k p_k^2$，熵 $=-\sum_k p_k\log_2 p_k$ —— 不纯度 / impurity
> - 信息增益 information gain —— 父节点不纯度 − 加权子节点不纯度

> 💡 **面试相关 / Interview-relevant**
> - "Gini 与熵的区别 / 信息增益"（★★★★★）
> - "决策树怎么防过拟合（剪枝/超参）"（★★★★★）
> - "为什么树不需要特征缩放"（★★★★）
> - "决策树为什么是高方差模型"（★★★★，引出 5.7 bagging）
> - "如何处理连续特征的分裂点"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解树如何用**不纯度(Gini/熵)**贪心选分裂。
   Understand how a tree greedily picks splits using **impurity (Gini/entropy)**.
2. **从零**实现一次最优分裂（看清 Gini 计算）。
   Implement one optimal split **from scratch**.
3. 理解**过拟合**与剪枝（深度/叶子/`ccp_alpha`）。
   Understand overfitting and pruning.
4. 读懂特征重要性。
   Read feature importances.
5. 理解树的高方差 → 为何需要森林(5.7)。
   See why trees are high-variance → motivating forests (5.7).

## 目录 / TOC
1. [先建直觉：一串是非题](#1)
2. [分裂准则：Gini 与熵 ⭐](#2)
3. [🚢 数据：Titanic](#3)
4. [从零：一次最优分裂 ⭐](#4)
5. [sklearn 树 + 可视化](#5)
6. [过拟合与剪枝 ⭐](#6)
7. [特征重要性 + 高方差](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：一串是非题 / Intuition First

决策树就像玩"二十个问题"猜身份：每次问一个能最有效区分的是非题，逐步缩小范围。预测泰坦尼克生还时，树可能先问"是女性吗？"，再问"是几等舱？"，一路问到底，叶子节点给出答案。
A decision tree is like playing "twenty questions": ask the most discriminating yes/no question each time, narrowing things down. To predict Titanic survival a tree might first ask "female?", then "which class?", and so on down to a leaf that gives the answer.

每个问题把当前这组人**切成两半**（如"票价 ≤ 30" vs "> 30"），目标是让切出来的两边**尽量纯**（一边大多生还、另一边大多遇难）。"纯不纯"用**不纯度**量化，下一节讲。
Each question splits the current group **in two** (e.g. "fare ≤ 30" vs "> 30"), aiming to make each side **as pure as possible** (one side mostly survivors, the other mostly not). "Purity" is quantified by **impurity**, next section.

因为树只是在比较"某特征是否 ≤ 某阈值"，**它从不关心特征的量纲**——所以决策树**不需要缩放**（和 KNN/SVM 相反）。
Since a tree only compares "is feature ≤ threshold", **it never cares about feature scale** — so trees **need no scaling** (unlike KNN/SVM).


<a id="2"></a>
## 2. 分裂准则：Gini 与熵 ⭐ / Splitting Criteria

树贪心地寻找"用哪个特征、哪个阈值切，能让两边子节点最纯"。纯度用**不纯度**衡量。设节点中各类别比例为 $p_k$：
The tree greedily searches for "which feature and threshold makes the children purest". Purity is measured by **impurity**. With class fractions $p_k$ in a node:

$$\text{Gini} = 1 - \sum_k p_k^2, \qquad \text{熵 Entropy} = -\sum_k p_k\log_2 p_k$$

两者都在**纯节点（全是一类）时为 0**，在**类别均匀分布时最大**。
Both are **0 at a pure node** (one class only) and **maximal when classes are evenly mixed**.

**信息增益** = 父节点不纯度 − 两个子节点不纯度的加权和。树选择**增益最大**的那个分裂。
**Information gain** = parent impurity − weighted sum of children impurities. The tree picks the split with the **largest gain**.

**Gini vs 熵**（面试常问）：结果通常极接近；Gini 不用算 log，**更快**（sklearn 默认）；熵更"信息论"。实务差别极小。
**Gini vs entropy** (often asked): results are usually almost identical; Gini avoids the log so it's **faster** (sklearn default); entropy is more "information-theoretic". The practical difference is tiny.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def gini(p): return 1 - (p**2).sum()        # p 是各类别比例向量
def entropy(p):
    p = p[p > 0]                            # 过滤掉 0(log 0 无定义)
    return -(p * np.log2(p)).sum()

# 二分类下, 让正类比例 p 从 0 变到 1, 看两种不纯度怎么变 / impurity vs class balance
ps = np.linspace(0.001, 0.999, 200)
g = [gini(np.array([p, 1-p])) for p in ps]      # 每个 p 对应类别分布 [p, 1-p]
e = [entropy(np.array([p, 1-p])) for p in ps]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, g, label="Gini (最大 0.5)"); ax.plot(ps, e, label="Entropy (最大 1.0)")
ax.set_xlabel("正类比例 positive-class fraction p"); ax.set_ylabel("不纯度 impurity"); ax.legend()
ax.set_title("不纯度: 纯(p=0或1)时=0, 均匀(p=0.5)时最大 / 0 when pure, max when 50/50")
plt.tight_layout(); plt.show()
print("两条曲线形状几乎一样 → Gini 与熵实务差别极小, sklearn 默认 Gini(更快)")


<a id="3"></a>
## 3. 数据：Titanic / The Titanic Dataset

泰坦尼克生还预测——分类教学里"另一个 Iris"。每行是一名乘客，标签 `survived`(0/1)。它混合了数值（年龄/票价）和类别（舱位/性别）特征，还有缺失值，很贴近真实。这里先做简化清洗。
Titanic survival prediction — the "other Iris" of classification teaching. Each row is a passenger, label `survived` (0/1). It mixes numeric (age/fare) and categorical (class/sex) features and has missing values — close to real life. We do a light cleanup first.


In [ ]:
df = sns.load_dataset("titanic")
print("Titanic:", df.shape, "| 生还率 survival rate", f"{df['survived'].mean():.0%}")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median())     # 年龄缺失用中位数填补(3.2)
d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)        # 性别编码成 0/1(3.5): male=1, female=0
print(d.head(3).to_string())

from sklearn.model_selection import train_test_split
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


<a id="4"></a>
## 4. 从零：一次最优分裂 ⭐ / Best Split From Scratch

完整建树代码较长，这里实现树每个节点真正在做的**核心动作**：扫描所有特征、所有候选阈值，用 Gini 加权选出增益最大的分裂。
A full tree builder is long; here we implement the **core action** each node performs: scan every feature and every candidate threshold, and pick the split with the largest Gini-weighted gain.


In [ ]:
def gini_impurity(y):
    if len(y) == 0: return 0
    p = np.bincount(y) / len(y)             # 各类别出现次数 / 总数 = 比例
    return 1 - (p**2).sum()

def best_split(X, y):
    n, d = X.shape
    parent = gini_impurity(y)               # 分裂前(父节点)的不纯度
    best = {"gain": -1}
    for j in range(d):                      # 遍历每个特征
        for thr in np.unique(X[:, j]):      # 遍历该特征所有可能的阈值
            left = X[:, j] <= thr           # 布尔数组: 哪些样本进入左子节点
            if left.sum() == 0 or left.sum() == n:
                continue                    # 跳过"全在一边"的无效分裂
            wl, wr = left.mean(), 1 - left.mean()       # 左/右子节点的样本占比(权重)
            # 子节点加权不纯度; ~left 是 left 的取反(右子节点)
            child = wl*gini_impurity(y[left]) + wr*gini_impurity(y[~left])
            gain = parent - child           # 信息增益 = 父不纯度 - 加权子不纯度
            if gain > best["gain"]:         # 记录增益最大的分裂
                best = {"gain": gain, "feat": j, "thr": thr}
    return best

bs = best_split(X_tr, y_tr)
print(f"根节点最优分裂 best root split: 特征 feature '{feat[bs['feat']]}' <= {bs['thr']}, 信息增益 gain {bs['gain']:.4f}")
print("(通常是 sex: 性别是 Titanic 最强预测因子 — 'women and children first')")


<a id="5"></a>
## 5. sklearn 树 + 可视化 / sklearn Tree & Visualization

`plot_tree` 能把整棵树画成流程图——这正是决策树最大的卖点：**可解释**。
`plot_tree` draws the whole tree as a flowchart — exactly the tree's biggest selling point: **interpretability**.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)   # 限制深度=3 便于看清
print(f"深度3 树 test 准确率 accuracy: {tree.score(X_te, y_te):.3f}")

fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(tree, feature_names=feat, class_names=["died","survived"],
          filled=True, rounded=True, fontsize=9, ax=ax)   # filled=按多数类着色
ax.set_title("Titanic 决策树(深度3) — 每个节点一个是/否问题 / each node = one yes/no question")
plt.tight_layout(); plt.show()
print("可解释性是树的最大卖点: 整个决策可读成流程图 / interpretability is the tree's killer feature")


<a id="6"></a>
## 6. 过拟合与剪枝 ⭐ / Overfitting & Pruning

不加限制的树会一直分裂，直到每个叶子都纯净——**完美拟合训练集**（训练准确率 100%）但**严重过拟合**（把噪声也学进去）。控制手段：
An unrestricted tree splits until every leaf is pure — **perfectly fitting the training set** (100% train accuracy) but **badly overfitting** (memorizing noise). Controls:
- **预剪枝 / pre-pruning**：`max_depth`、`min_samples_leaf`、`min_samples_split`（边长边限制）。
  limit growth via `max_depth`, `min_samples_leaf`, `min_samples_split`.
- **后剪枝 / post-pruning**：`ccp_alpha`（代价复杂度剪枝，先长满再剪掉弱分支，类似 4.5 Lasso 的复杂度惩罚）。
  `ccp_alpha` (cost-complexity pruning: grow full, then prune weak branches; like Lasso's penalty, 4.5).


In [ ]:
depths = range(1, 21)
tr_acc, te_acc = [], []
for dep in depths:                          # 让最大深度从 1 增到 20
    t = DecisionTreeClassifier(max_depth=dep, random_state=0).fit(X_tr, y_tr)
    tr_acc.append(t.score(X_tr, y_tr))      # 训练准确率
    te_acc.append(t.score(X_te, y_te))      # 测试准确率

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, tr_acc, "o-", label="训练 train")
ax.plot(depths, te_acc, "s-", label="测试 test")
ax.set_xlabel("max_depth"); ax.set_ylabel("准确率 accuracy"); ax.legend()
ax.set_title("过拟合: 深度↑ 训练→1, 测试先升后降 / deeper = train→1 but test peaks then drops")
plt.tight_layout(); plt.show()
# 不限制深度的"满树": 训练准确率几乎 100% = 过拟合
print(f"满树训练准确率 full-tree train accuracy: {DecisionTreeClassifier(random_state=0).fit(X_tr,y_tr).score(X_tr,y_tr):.3f} (≈完美=过拟合)")
print(f"最佳测试深度约 best test depth ≈ {list(depths)[int(np.argmax(te_acc))]}")


In [ ]:
# 后剪枝: ccp_alpha 路径 / cost-complexity pruning path
# cost_complexity_pruning_path 给出一系列候选 alpha(剪枝强度), 从0(满树)到很大(只剩根)
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_tr, y_tr)
alphas = path.ccp_alphas[:-1]               # 去掉最后一个(会把树剪到只剩根节点)
# 对每个 alpha 训练一棵剪枝后的树, 看测试准确率
scores = [DecisionTreeClassifier(ccp_alpha=a, random_state=0).fit(X_tr,y_tr).score(X_te,y_te)
          for a in alphas]
best_a = alphas[int(np.argmax(scores))]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, scores, marker=".")
ax.axvline(best_a, color="r", ls="--", label=f"最佳 best ccp_alpha={best_a:.4f}")
ax.set_xlabel("ccp_alpha (剪枝强度 pruning strength)"); ax.set_ylabel("test 准确率 accuracy"); ax.legend()
ax.set_title("后剪枝: 增大 alpha 剪掉弱分支, 防过拟合 / larger alpha prunes weak branches")
plt.tight_layout(); plt.show()
print(f"最佳 best ccp_alpha={best_a:.4f}, test 准确率 {max(scores):.3f}")


<a id="7"></a>
## 7. 特征重要性 + 高方差 / Importance & High Variance

**特征重要性** = 该特征在所有分裂中带来的总不纯度下降（归一化后）。它比线性系数更适合非线性模型。
**Feature importance** = total impurity decrease contributed by a feature across all splits (normalized). More suitable than linear coefficients for nonlinear models.

**树是高方差模型**：训练数据稍有扰动，就可能长出完全不同的树。下面在不同的 bootstrap 子样本上各训一棵满树，看它们对同一批测试点的预测有多不一致——这正是 **5.7 随机森林用 bagging 平均掉方差** 的动机。
**Trees are high-variance:** a small change in training data can grow a very different tree. Below we train full trees on different bootstrap subsamples and see how much they disagree on the same test points — exactly why **Random Forest (5.7) averages many trees to cancel the variance**.


In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
imp = pd.Series(tree.feature_importances_, index=feat).sort_values(ascending=False)
print("特征重要性 feature importance:"); print(imp.round(3).to_string())

# 高方差演示: 在 30 个 bootstrap 子样本上各训一棵满树, 看预测分歧 / prediction instability
rng = np.random.default_rng(0)
preds = []
for _ in range(30):
    # bootstrap: 有放回随机抽 n 个样本(会重复), 模拟"训练数据的小扰动"
    idx = rng.choice(len(X_tr), len(X_tr), replace=True)
    t = DecisionTreeClassifier(random_state=0).fit(X_tr[idx], y_tr[idx])
    preds.append(t.predict(X_te))
preds = np.array(preds)                       # 形状 (30 棵树, 测试样本数)
frac1 = preds.mean(0)                          # 每个测试点被预测为 1 的比例(跨30棵树)
# 比例落在 20%~80% 说明这些树对该点意见分歧大(不稳定)
unstable = ((frac1 > 0.2) & (frac1 < 0.8)).mean()
print(f"\n30 棵 bootstrap 满树: {unstable:.0%} 的测试样本预测在树间摇摆 / swing between trees")
print(f"单棵满树 test 准确率波动 std: {preds.mean(1).std():.3f}")
print("→ 满树对数据扰动敏感(高方差); 5.7 用 bagging 平均多棵树降方差 / forests average it away")


<a id="8"></a>
## 8. 小结 / Summary

```
决策树: 贪心选分裂, 最大化信息增益(父-加权子不纯度); Gini/熵实务等价(Gini 更快)
轴对齐方块边界; 可解释性强; 不需缩放; 处理混合类型
过拟合: 满树训练准确率→100%; 预剪枝(max_depth/min_samples) + 后剪枝(ccp_alpha)
特征重要性 = 总不纯度下降; 树是高方差模型 → 引出 bagging(5.7)
```

### 💡 面试速查 / Interview cheat-sheet
1. **Gini=1-Σp², 熵=-Σp·log p**；信息增益选分裂；Gini 更快(默认)。
   Gini=1-Σp², entropy=-Σp·log p; gain picks splits; Gini is faster (default).
2. **防过拟合**：max_depth / min_samples_leaf / ccp_alpha。
   Curb overfitting with max_depth / min_samples_leaf / ccp_alpha.
3. **不需缩放** —— 分裂只比较阈值，与量纲无关。
   No scaling needed — splits only compare thresholds.
4. **高方差** —— 数据小扰动→不同树 → 森林(bagging)平均降方差。
   High variance — small data changes → different trees → forests (bagging) average it down.
5. **特征重要性**基于不纯度下降，注意对高基数特征有偏。
   Importance is impurity-based; beware bias toward high-cardinality features.

### 下一节 / Next
**5.7 随机森林**——把很多高方差的树 bagging + 特征随机，平均掉方差，几乎不需调参的强力开箱即用模型。
**5.7 Random Forest** — bagging many high-variance trees plus feature randomness to cancel variance; a powerful near-zero-tuning baseline.
